In [1]:
import os
print("Current folder:", os.getcwd())

Current folder: /tmp/final2/student_a_intracellular/notebooks


In [2]:
import pandas as pd
import sys
sys.path.insert(0, "../src")
from imodulon_loader import IcaData   # same container as pymodulon.core.IcaData (see src/imodulon_loader.py)

from imodulon_loader import DEFAULT_BASE as base, DEFAULT_TRN_BASE as trn_base   # filled by `python get_data.py`

M = pd.read_csv(f"{base}/M.csv", index_col=0)
A = pd.read_csv(f"{base}/A.csv", index_col=0)
sample_table = pd.read_csv(f"{base}/metadata_qc.csv", index_col=0)
trn = pd.read_csv(f"{trn_base}/TRN.csv")

print("M:", M.shape)
print("A:", A.shape)
print("sample_table:", sample_table.shape)
print("trn:", trn.shape)

M: (4257, 201)
A: (201, 1035)
sample_table: (1035, 42)
trn: (10792, 6)


In [3]:
# First attempt with the files as published: this fails on purpose, showing the name mismatch fixed below.
try:
    ica_data = IcaData(M, A, gene_table=None, sample_table=sample_table, trn=trn)

    print("Total iModulons:", len(ica_data.imodulon_names))
    print()
    crp_names = [name for name in ica_data.imodulon_names if "crp" in str(name).lower()]
    print("iModulons containing 'crp':", crp_names)
except ValueError as e:
    print("ValueError:", e)


ValueError: M and A matrices have different iModulon names


In [4]:
print("M columns (first 5):", M.columns[:5].tolist())
print("A index (first 5):", A.index[:5].tolist())
print()
print("Same set of names?", set(M.columns) == set(A.index))
print("Same order?", M.columns.tolist() == A.index.tolist())

M columns (first 5): ['Sugar Diacid', 'Translation', 'ygdT', 'OxyR', 'FlhDC-2']
A index (first 5): ['Sugar Diacid', 'Translation', 'ygdT', 'OxyR', 'FlhDC-2']

Same set of names? False
Same order? False


In [5]:
only_in_M = set(M.columns) - set(A.index)
only_in_A = set(A.index) - set(M.columns)

print("In M but not A:", only_in_M)
print("In A but not M:", only_in_A)
print()
print("Number of M columns:", len(M.columns))
print("Number of A index:", len(A.index))

In M but not A: {'Superoxide'}
In A but not M: {'SoxS'}

Number of M columns: 201
Number of A index: 201


In [6]:
m_pos = M.columns.get_loc('Superoxide')
a_pos = A.index.get_loc('SoxS')

print("Position of 'Superoxide' in M:", m_pos)
print("Position of 'SoxS' in A:", a_pos)

Position of 'Superoxide' in M: 22
Position of 'SoxS' in A: 22


In [7]:
M = M.rename(columns={'Superoxide': 'SoxS'})

print("Same order now?", M.columns.tolist() == A.index.tolist())

Same order now? True


In [8]:
ica_data = IcaData(M, A, gene_table=None, sample_table=sample_table, trn=trn)

print("Total iModulons:", len(ica_data.imodulon_names))
print()
crp_names = [name for name in ica_data.imodulon_names if "crp" in str(name).lower()]
print("iModulons containing 'crp':", crp_names)

Total iModulons: 201

iModulons containing 'crp': ['crp KO-1', 'Crp-1', 'Crp-2', 'crp KO-2']


In [9]:
crp1_activity = ica_data.A.loc["Crp-1"]
crp2_activity = ica_data.A.loc["Crp-2"]

activity_df = pd.DataFrame({
    "Crp-1": crp1_activity,
    "Crp-2": crp2_activity
})

print(activity_df.shape)
activity_df.head()

(1035, 2)


,Crp-1,Crp-2
p1k_00001,0.133712,0.043984
p1k_00002,-0.133712,-0.043984
p1k_00003,-1.801538,-11.224692
p1k_00004,-1.789236,-13.024489
p1k_00005,-1.277517,-0.154498


In [10]:
print(sample_table.columns.tolist())

['sample_id', 'study', 'project', 'condition', 'rep_id', 'Strain Description', 'Strain', 'Culture Type', 'Evolved Sample', 'Base Media', 'Temperature (C)', 'pH', 'Carbon Source (g/L)', 'Nitrogen Source (g/L)', 'Electron Acceptor', 'Trace Element Mixture', 'Supplement', 'Antibiotic for selection', 'Growth Rate (1/hr)', 'Isolate Type', 'Additional Details', 'project_reference', 'Sequencing Machine', 'LibraryLayout', 'Platform', 'Biological Replicates', 'DOI', 'GEO', 'SRX', 'Run', 'R1', 'R2', 'contact', 'creator', 'passed_fastqc', 'passed_pct_reads_mapped', 'passed_reads_mapped_to_CDS', 'passed_global_correlation', 'full_name', 'passed_similar_replicates', 'passed_number_replicates', 'run_date']


In [11]:
crp_mask = sample_table["Strain Description"].astype(str).str.contains("crp", case=False, na=False) | \
           sample_table["Strain"].astype(str).str.contains("crp", case=False, na=False)

crp_samples = sample_table[crp_mask]
print("Number of CRP-related samples:", len(crp_samples))
print()
print(crp_samples[["sample_id", "Strain", "Strain Description", "project", "condition"]])

Number of CRP-related samples: 17

                        sample_id  Strain  \
p1k_00108     crp__delar1_glyc__1  MG1655   
p1k_00109     crp__delar1_glyc__2  MG1655   
p1k_00110  crp__delar1ar2_glyc__1  MG1655   
p1k_00111  crp__delar1ar2_glyc__2  MG1655   
p1k_00112  crp__delar1ar2_glyc__3  MG1655   
p1k_00113     crp__delar2_glyc__1  MG1655   
p1k_00114     crp__delar2_glyc__2  MG1655   
p1k_00115     crp__delar2_glyc__3  MG1655   
p1k_00116      crp__delcrp_fru__1  MG1655   
p1k_00117      crp__delcrp_fru__2  MG1655   
p1k_00118      crp__delcrp_fru__3  MG1655   
p1k_00119      crp__delcrp_glc__1  MG1655   
p1k_00120      crp__delcrp_glc__2  MG1655   
p1k_00121      crp__delcrp_glc__3  MG1655   
p1k_00122     crp__delcrp_glyc__1  MG1655   
p1k_00123     crp__delcrp_glyc__2  MG1655   
p1k_00124     crp__delcrp_glyc__3  MG1655   

                                      Strain Description project  \
p1k_00108        Escherichia coli K-12 MG1655 crp_delAr1     crp   
p1k_00109        E

In [12]:
def categorize(strain_desc):
    s = str(strain_desc).lower()
    if "delar1delar2" in s.replace(" ", ""):
        return "CRP ΔAr1+Ar2"
    elif "delar2" in s:
        return "CRP ΔAr2 (Class II)"
    elif "delar1" in s:
        return "CRP ΔAr1 (Class I)"
    elif "del_crp" in s:
        return "CRP full deletion"
    else:
        return "Other/WT"

sample_table["crp_category"] = sample_table["Strain Description"].apply(categorize)

merged = activity_df.join(sample_table[["crp_category", "condition"]])
print(merged["crp_category"].value_counts())
merged.head()

crp_category
Other/WT               1018
CRP full deletion         9
CRP ΔAr1+Ar2              3
CRP ΔAr2 (Class II)       3
CRP ΔAr1 (Class I)        2
Name: count, dtype: int64


,Crp-1,Crp-2,crp_category,condition
p1k_00001,0.133712,0.043984,Other/WT,wt_glc
p1k_00002,-0.133712,-0.043984,Other/WT,wt_glc
p1k_00003,-1.801538,-11.224692,Other/WT,wt_dpd
p1k_00004,-1.789236,-13.024489,Other/WT,wt_dpd
p1k_00005,-1.277517,-0.154498,Other/WT,wt_fe


In [13]:
import matplotlib.pyplot as plt

colors = {
    "Other/WT": "lightgray",
    "CRP full deletion": "blue",
    "CRP ΔAr1 (Class I)": "green",
    "CRP ΔAr2 (Class II)": "orange",
    "CRP ΔAr1+Ar2": "red"
}

fig, ax = plt.subplots(figsize=(8, 6))

for category, color in colors.items():
    subset = merged[merged["crp_category"] == category]
    alpha = 0.3 if category == "Other/WT" else 0.9
    size = 15 if category == "Other/WT" else 60
    ax.scatter(subset["Crp-1"], subset["Crp-2"], label=category, color=color, alpha=alpha, s=size, edgecolor="black" if category != "Other/WT" else "none")

ax.set_xlabel("Crp-1 Activity")
ax.set_ylabel("Crp-2 Activity")
ax.set_title("Crp-1 vs Crp-2 iModulon Activity (reproduction of Fig. 4B)")
ax.legend()
ax.axhline(0, color="black", linewidth=0.5, linestyle="--")
ax.axvline(0, color="black", linewidth=0.5, linestyle="--")

plt.tight_layout()
plt.savefig("../outputs/a1/condition_response_curve.png", dpi=150)
plt.show()

In [14]:
output = merged.join(sample_table.drop(columns=["crp_category", "condition"]))
output.to_csv("../outputs/a1/crp_activity_by_condition.csv")
print("Saved:", output.shape)

Saved: (1035, 45)


In [15]:
import sys
sys.path.append("../src")

from imodulon_loader import build_ica_data, get_activity_by_condition
from condition_response import categorize_crp_strains, get_condition_response

test_ica = build_ica_data()
test_df = get_condition_response(test_ica, "Crp-1", "Crp-2", category_fn=categorize_crp_strains)
print(test_df["category"].value_counts())

category
Other/WT               1018
CRP full deletion         9
CRP ΔAr1+Ar2              3
CRP ΔAr2 (Class II)       3
CRP ΔAr1 (Class I)        2
Name: count, dtype: int64


In [16]:
rpos_names = [name for name in ica_data.imodulon_names if "rpos" in str(name).lower()]
print("iModulons containing 'rpos':", rpos_names)
print("iModulons containing 'crp':", crp_names)

iModulons containing 'rpos': ['RpoS']
iModulons containing 'crp': ['crp KO-1', 'Crp-1', 'Crp-2', 'crp KO-2']


In [17]:
crp_project_mask = sample_table["project"].astype(str).str.contains("crp", case=False, na=False)
rpos_project_mask = sample_table["project"].astype(str).str.contains("rpos", case=False, na=False)

print("Samples in CRP-related projects:", crp_project_mask.sum())
print("Samples in RpoS-related projects:", rpos_project_mask.sum())
print()
print("Unique CRP conditions:", sample_table.loc[crp_project_mask, "condition"].nunique())
print("Unique RpoS conditions:", sample_table.loc[rpos_project_mask, "condition"].nunique())

Samples in CRP-related projects: 30
Samples in RpoS-related projects: 0

Unique CRP conditions: 12
Unique RpoS conditions: 0


In [18]:
print(sorted(sample_table["project"].dropna().unique().tolist()))

['42c', 'DDB2', 'DDB3', 'abx_media', 'acid', 'acid_tale', 'adnB', 'cck', 'cck_gnd', 'cck_pgi', 'cck_pts', 'cck_sdh', 'cck_tpiA', 'control', 'cra_crp', 'crp', 'eep', 'efeU', 'fps', 'fur', 'glu', 'hot_tale', 'ica', 'minicoli', 'minspan', 'misc', 'misc2', 'nac_ntrc', 'nquinone', 'omics', 'ompr', 'oxidative', 'oxyR', 'pal', 'pcoli', 'pdhr_ale', 'quinone', 'ros_tale', 'rpoB', 'ssw', 'svns_pgi', 'tcs', 'ytf', 'ytf2', 'ytf3']


In [19]:
print(trn.columns.tolist())
print()
print(trn.head())

['regulator', 'gene_name', 'gene_id', 'effect', 'evidence', 'source']

  regulator gene_name gene_id effect  evidence           source
0      AccB      accB   b3255      -       0.0     RegulonDB_TF
1      RpoD      pheU   b4134      +       0.0  RegulonDB_sigma
2      RpoD      pheP   b0576      +       0.0  RegulonDB_sigma
3      RpoD      pheL   b2598      +       0.0  RegulonDB_sigma
4      RpoD      pheA   b2599      +       0.0  RegulonDB_sigma


In [20]:
crp_targets = trn[trn["regulator"] == "Crp"]
print("Number of CRP target-gene edges:", len(crp_targets))
print()
print(crp_targets.head(10))

Number of CRP target-gene edges: 556

     regulator gene_name gene_id effect  evidence        source
3747       Crp      gutM   b2706      +       0.0  RegulonDB_TF
3750       Crp      grpE   b2614      +       0.0  RegulonDB_TF
3751       Crp      grcA   b2579      +       0.0  RegulonDB_TF
3753       Crp      gntU   b4476      +       0.0  RegulonDB_TF
3756       Crp      gntK   b3437      +       0.0  RegulonDB_TF
3761       Crp      gltA   b0720      +       0.0  RegulonDB_TF
3762       Crp      glpX   b3925      +       0.0  RegulonDB_TF
3768       Crp      gutQ   b2708      +       0.0  RegulonDB_TF
3770       Crp      gyrA   b2231      +       0.0  RegulonDB_TF
3789       Crp      hicB   b1438      +       0.0  RegulonDB_TF


In [21]:
# Binary flag: was this sample a CRP genetic perturbation (any type)?
sample_table["crp_perturbed"] = sample_table["Strain Description"].astype(str).str.contains("crp", case=False, na=False)

print("Perturbed samples:", sample_table["crp_perturbed"].sum())
print("WT/other samples:", (~sample_table["crp_perturbed"]).sum())

Perturbed samples: 17
WT/other samples: 1018


In [22]:
# Find CRP's own gene_id from TRN (regulators are also genes, so they may appear as targets elsewhere too)
print(trn[trn["gene_name"].str.lower() == "crp"])

      regulator gene_name gene_id effect  evidence           source
1077       RpoD       crp   b3357      +       0.0  RegulonDB_sigma
5903        Crp       crp   b3357      ?       1.0     RegulonDB_TF
5944        Cra       crp   b3357      +       1.0     RegulonDB_TF
7856        Fis       crp   b3357      -       1.0     RegulonDB_TF
10392     ppGpp       crp   b3357    NaN       NaN           Ecocyc


In [23]:
log_tpm_qc = pd.read_csv("../data/precise1k/precise1k/data/precise1k/log_tpm_qc.csv", index_col=0)
print("log_tpm_qc shape:", log_tpm_qc.shape)
print(log_tpm_qc.index[:5].tolist())

log_tpm_qc shape: (4257, 1035)
['b0002', 'b0003', 'b0004', 'b0005', 'b0006']


In [24]:
crp_expression = log_tpm_qc.loc["b3357"]

print("CRP expression shape:", crp_expression.shape)
print(crp_expression.head())
print()

example_target = crp_targets.iloc[0]
print("Example target gene:", example_target["gene_name"], "/", example_target["gene_id"])


CRP expression shape: (1035,)
p1k_00001    9.864955
p1k_00002    9.839998
p1k_00003    9.930051
p1k_00004    9.761794
p1k_00005    9.821963
Name: b3357, dtype: float64

Example target gene: gutM / b2706


In [25]:
target_expression = log_tpm_qc.loc["b2706"]  # gutM

test_df = pd.DataFrame({
    "target_expr": target_expression,
    "crp_expr": crp_expression,
    "perturbed": sample_table["crp_perturbed"].astype(int)
})

print(test_df.shape)
test_df.head()

(1035, 3)


,target_expr,crp_expr,perturbed
p1k_00001,3.075557,9.864955,0
p1k_00002,2.990521,9.839998,0
p1k_00003,2.993601,9.930051,0
p1k_00004,3.574856,9.761794,0
p1k_00005,2.702236,9.821963,0


In [26]:
import statsmodels.api as sm

# --- Step 1: Raw effect (target ~ perturbed) ---
X_raw = sm.add_constant(test_df["perturbed"])
model_raw = sm.OLS(test_df["target_expr"], X_raw).fit()

print("=== RAW EFFECT (target ~ perturbed) ===")
print("Perturbation coefficient:", model_raw.params["perturbed"])
print("p-value:", model_raw.pvalues["perturbed"])
print()

# --- Step 2: Conditioned effect (target ~ perturbed + crp_expr) ---
X_cond = sm.add_constant(test_df[["perturbed", "crp_expr"]])
model_cond = sm.OLS(test_df["target_expr"], X_cond).fit()

print("=== CONDITIONED EFFECT (target ~ perturbed + crp_expr) ===")
print("Perturbation coefficient:", model_cond.params["perturbed"])
print("p-value:", model_cond.pvalues["perturbed"])
print()

# --- Step 3: Interpretation ---
raw_significant = model_raw.pvalues["perturbed"] < 0.05
cond_significant = model_cond.pvalues["perturbed"] < 0.05

if raw_significant and not cond_significant:
    verdict = "CAUSAL (mediated by CRP) — effect disappears after conditioning"
elif raw_significant and cond_significant:
    verdict = "NOT fully mediated by CRP — effect persists after conditioning"
elif not raw_significant:
    verdict = "NO raw effect to begin with — inconclusive/no relationship"

print("VERDICT:", verdict)

=== RAW EFFECT (target ~ perturbed) ===
Perturbation coefficient: 0.4299840056437297
p-value: 0.1051836133964172

=== CONDITIONED EFFECT (target ~ perturbed + crp_expr) ===
Perturbation coefficient: 1.3753669146416219
p-value: 4.159194290666792e-05

VERDICT: NO raw effect to begin with — inconclusive/no relationship


In [27]:
sample_table["crp_category"] = sample_table["Strain Description"].apply(categorize_crp_strains)
print(sample_table["crp_category"].value_counts())

crp_category
Other/WT               1018
CRP full deletion         9
CRP ΔAr1+Ar2              3
CRP ΔAr2 (Class II)       3
CRP ΔAr1 (Class I)        2
Name: count, dtype: int64


In [28]:
target_expression = log_tpm_qc.loc["b2706"]  # gutM
crp_expression = log_tpm_qc.loc["b3357"]      # CRP itself

test_df2 = pd.DataFrame({
    "target_expr": target_expression,
    "crp_expr": crp_expression,
    "category": sample_table["crp_category"]
})

# Compare each perturbation category's mean gutM expression against WT/Other
wt_mean = test_df2.loc[test_df2["category"] == "Other/WT", "target_expr"].mean()

print("WT/Other mean gutM expression:", wt_mean)
print()

for cat in ["CRP full deletion", "CRP \u0394Ar1 (Class I)", "CRP \u0394Ar2 (Class II)", "CRP \u0394Ar1+Ar2"]:
    subset = test_df2[test_df2["category"] == cat]
    print(f"{cat}: n={len(subset)}, mean target_expr={subset['target_expr'].mean():.3f}, mean crp_expr={subset['crp_expr'].mean():.3f}")

WT/Other mean gutM expression: 2.719976272768031

CRP full deletion: n=9, mean target_expr=3.121, mean crp_expr=0.206
CRP ΔAr1 (Class I): n=2, mean target_expr=3.354, mean crp_expr=10.727
CRP ΔAr2 (Class II): n=3, mean target_expr=3.000, mean crp_expr=11.175
CRP ΔAr1+Ar2: n=3, mean target_expr=3.250, mean crp_expr=10.159


In [29]:
print("Total CRP targets:", len(crp_targets))
print()
print("Evidence value counts:")
print(crp_targets["evidence"].value_counts())
print()
print("Effect direction counts:")
print(crp_targets["effect"].value_counts())
print()
# Show a broader sample of targets, sorted by evidence score (higher = more confident)
print(crp_targets.sort_values("evidence", ascending=False)[["gene_name", "gene_id", "effect", "evidence", "source"]].head(20))

Total CRP targets: 556

Evidence value counts:
evidence
1.0    299
0.0    231
Name: count, dtype: int64

Effect direction counts:
effect
+    395
-     87
?     48
Name: count, dtype: int64

     gene_name gene_id effect  evidence        source
8175      xseA   b2509      -       1.0  RegulonDB_TF
8173      xylE   b4031      +       1.0  RegulonDB_TF
8171      ychH   b1205      +       1.0  RegulonDB_TF
8155      ygeA   b2840      +       1.0  RegulonDB_TF
8154      yhcH   b3221      +       1.0  RegulonDB_TF
8137      uxuA   b4322      +       1.0  RegulonDB_TF
8136      uxuB   b4323      +       1.0  RegulonDB_TF
8120      yhfA   b3356      ?       1.0  RegulonDB_TF
8081      yiaJ   b3574      -       1.0  RegulonDB_TF
6061      csqR   b3884      ?       1.0  RegulonDB_TF
6062      deoD   b4384      ?       1.0  RegulonDB_TF
6063      glnG   b3868      ?       1.0  RegulonDB_TF
6064      fucI   b2802      +       1.0  RegulonDB_TF
6065      fucP   b2801      +       1.0  RegulonDB_TF

In [30]:
print(crp_targets[crp_targets["gene_name"].str.contains("lac", case=False, na=False)])


     regulator gene_name gene_id effect  evidence        source
5566       Crp      ulaC   b4195      +       1.0  RegulonDB_TF
5779       Crp      lacZ   b0344      ?       1.0  RegulonDB_TF
5780       Crp      lacY   b0343      ?       1.0  RegulonDB_TF
5781       Crp      lacI   b0345      +       1.0  RegulonDB_TF
5782       Crp      lacA   b0342      ?       1.0  RegulonDB_TF
